### Prepare dataset

In [1]:
import pandas as pd
import numpy as np


weather_path = "data/weather.csv"
cab_path = "data/cab_rides.csv"

In [2]:
df_cab = pd.read_csv(cab_path, on_bad_lines="skip", low_memory=False)
df_weather = pd.read_csv(weather_path, on_bad_lines="skip", low_memory=False)

### KNN Imputation
For missing rain values, use a KNN-based regression/imputation approach with the numeric predictors like temp, clouds, pressure, humidity, and wind. KNNImputer fills missing values by using the mean of the nearest rows.

In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsRegressor

df_weather = df_weather.copy()

target = 'rain'

# One-hot encode location
df_weather = pd.get_dummies(df_weather, columns=['location'], drop_first=False)

feature_cols = [col for col in df_weather.columns if col != target and col != 'time_stamp']

train_df = df_weather[df_weather[target].notna()].copy()
missing_df = df_weather[df_weather[target].isna()].copy()

X_train = train_df[feature_cols]
y_train = train_df[target]
X_missing = missing_df[feature_cols]

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_missing_scaled = scaler.transform(X_missing)

knn = KNeighborsRegressor(n_neighbors=5, weights='distance')
knn.fit(X_train_scaled, y_train)

df_weather.loc[df_weather[target].isna(), target] = knn.predict(X_missing_scaled)

In [ ]:
# Convert timestamps (assuming milliseconds)
df_cab['time_stamp'] = pd.to_datetime(df_cab['time_stamp'], unit='ms').astype('datetime64[ns]')
df_weather['time_stamp'] = pd.to_datetime(df_weather['time_stamp'], unit='s').astype('datetime64[ns]')

df_cab = df_cab.sort_values('time_stamp')
df_weather = df_weather.sort_values('time_stamp')

# Merge cab trip with weather information with nearest timestamp within 15 minutes
merged = pd.merge_asof(
    df_cab,
    df_weather,
    on='time_stamp',
    direction='nearest',
    tolerance=pd.Timedelta(minutes=15)
)


# Drop rows where no weather match
merged = merged.dropna()

### Feature Engineering

In [5]:
# Time features
merged['hour'] = merged['time_stamp'].dt.hour
merged['day'] = merged['time_stamp'].dt.day
merged['weekday'] = merged['time_stamp'].dt.weekday

# Example categorical encoding
merged = pd.get_dummies(merged, columns=['cab_type', 'source', 'destination', 'name'], drop_first=False)

# Drop unused
merged = merged.drop(columns=['time_stamp', 'id', 'product_id'])

### Prepare Data

In [6]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# Target
y = merged['price']
X = merged.drop(columns=['price'])

# Scale
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Split
X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y, test_size=0.2, random_state=42
)

### Train ML models - Linear Regression & Random Forrest

In [7]:
from sklearn.linear_model import LinearRegression

lr = LinearRegression()
lr.fit(X_train, y_train)

y_pred_lr = lr.predict(X_test)

In [8]:
from xgboost import XGBRFRegressor

rf = XGBRFRegressor(
    n_estimators=100,
    max_depth=10,
    random_state=42,
    device="cuda",
    tree_method="hist"
)

rf.fit(X_train, y_train)
y_pred_rf = rf.predict(X_test)

/home/markphan24/uber-lyft-prices/.venv/lib/python3.12/site-packages/xgboost/core.py:751: UserWarning: [17:38:37] WARNING: /__w/xgboost/xgboost/src/common/error_msg.cc:62: Falling back to prediction using DMatrix due to mismatched devices. This might lead to higher memory usage and slower performance. XGBoost is running on: cuda:0, while the input data is on: cpu.
Potential solutions:
- Use a data structure that matches the device ordinal in the booster.
- Set the device for booster before call to inplace_predict.

This warning will only be shown once.

  return func(**kwargs)


In [9]:
import torch
import torch.nn as nn

class CabPriceModel(nn.Module):
    def __init__(self, input_dim):
        super().__init__()
        self.model = nn.Sequential(
            nn.Linear(input_dim, 1024),
            nn.ReLU(),
            nn.BatchNorm1d(1024),

            nn.Linear(1024, 512),
            nn.ReLU(),
            nn.Dropout(0.2),

            nn.Linear(512, 32),
            nn.ReLU(),

            nn.Linear(32, 1)
        )

    def forward(self, x):
        return self.model(x)

In [ ]:
import os
import torch
import torch.nn as nn

checkpoint_path = "nn_checkpoint.pth"

# Select device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

# Convert to tensors and move to device
X_train_t = torch.tensor(X_train, dtype=torch.float32).to(device)
y_train_t = torch.tensor(y_train.values, dtype=torch.float32).view(-1, 1).to(device)

X_test_t = torch.tensor(X_test, dtype=torch.float32).to(device)
y_test_t = torch.tensor(y_test.values, dtype=torch.float32).view(-1, 1).to(device)

# Model
model = CabPriceModel(X_train.shape[1]).to(device)

# Loss + optimizer
criterion = nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

start_epoch = 0
best_loss = float("inf")

# Load checkpoint if it exists
if os.path.exists(checkpoint_path):
    checkpoint = torch.load(checkpoint_path, map_location=device)
    model.load_state_dict(checkpoint["model_state_dict"])
    optimizer.load_state_dict(checkpoint["optimizer_state_dict"])
    start_epoch = checkpoint["epoch"] + 1
    best_loss = checkpoint.get("loss", float("inf"))
    print(f"Loaded checkpoint from epoch {checkpoint['epoch']}")

num_epochs = 800

for epoch in range(start_epoch, num_epochs):
    model.train()

    optimizer.zero_grad()
    outputs = model(X_train_t)
    loss = criterion(outputs, y_train_t)

    loss.backward()
    optimizer.step()

    if epoch % 10 == 0:
        print(f"Epoch {epoch}, Loss: {loss.item():.4f}")

        torch.save({
            "epoch": epoch,
            "model_state_dict": model.state_dict(),
            "optimizer_state_dict": optimizer.state_dict(),
            "loss": loss.item()
        }, checkpoint_path)

# Final save
torch.save({
    "epoch": epoch,
    "model_state_dict": model.state_dict(),
    "optimizer_state_dict": optimizer.state_dict(),
    "loss": loss.item()
}, checkpoint_path)

print("Training complete and checkpoint saved.")

Using device: cuda
Epoch 0, Loss: 353.9854
Epoch 10, Loss: 159.6431
Epoch 20, Loss: 37.1792
Epoch 30, Loss: 13.2660
Epoch 40, Loss: 6.9129
Epoch 50, Loss: 5.3787
Epoch 60, Loss: 4.6920
Epoch 70, Loss: 4.3283
Epoch 80, Loss: 4.0924
Epoch 90, Loss: 3.9448
Epoch 100, Loss: 3.8693
Epoch 110, Loss: 3.8250
Epoch 120, Loss: 3.7822
Epoch 130, Loss: 3.7429
Epoch 140, Loss: 3.7140
Epoch 150, Loss: 3.6757
Epoch 160, Loss: 3.6478
Epoch 170, Loss: 3.6216
Epoch 180, Loss: 3.6088
Epoch 190, Loss: 3.5799
Epoch 200, Loss: 3.5628
Epoch 210, Loss: 3.5442
Epoch 220, Loss: 3.5283
Epoch 230, Loss: 3.5141
Epoch 240, Loss: 3.4986
Epoch 250, Loss: 3.4783
Epoch 260, Loss: 3.4545
Epoch 270, Loss: 3.4499
Epoch 280, Loss: 3.4351
Epoch 290, Loss: 3.4156
Epoch 300, Loss: 3.4136
Epoch 310, Loss: 3.3986
Epoch 320, Loss: 3.3829
Epoch 330, Loss: 3.3771
Epoch 340, Loss: 3.3542
Epoch 350, Loss: 3.3436
Epoch 360, Loss: 3.3458
Epoch 370, Loss: 3.3197
Epoch 380, Loss: 3.3200
Epoch 390, Loss: 3.3125
Epoch 400, Loss: 3.3105
Ep

### 500 epochs

In [ ]:
import numpy as np

checkpoint_path = "nn_checkpoint.pth"
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
checkpoint = torch.load(checkpoint_path, map_location=device)
model.load_state_dict(checkpoint["model_state_dict"])
model.eval()
with torch.no_grad():
    y_preds_nn = model(X_test_t).detach().cpu().numpy()
    y_true_nn = y_test_t.detach().cpu().numpy()

In [15]:
from sklearn.metrics import root_mean_squared_error, r2_score, mean_absolute_error
import numpy as np


def bootstrap_metric_ci_pm(y_true, y_pred, metric_func, n_bootstrap=500, random_state=42):
    rng = np.random.default_rng(random_state)

    y_true = np.array(y_true)
    y_pred = np.array(y_pred)

    n = len(y_true)
    scores = []

    for _ in range(n_bootstrap):
        idx = rng.integers(0, n, n)
        y_true_sample = y_true[idx]
        y_pred_sample = y_pred[idx]
        scores.append(metric_func(y_true_sample, y_pred_sample))

    mean_score = np.mean(scores)
    lower = np.percentile(scores, 2.5)
    upper = np.percentile(scores, 97.5)
    pm = (upper - lower) / 2

    return mean_score, pm

In [16]:

models = {
    "Linear Regression": (y_test, y_pred_lr),
    "Random Forest": (y_test, y_pred_rf),
    "Neural Network": (y_true_nn, y_preds_nn),
}

metrics = {
    "RMSE": root_mean_squared_error,
    "R2": r2_score,
    "MAE": mean_absolute_error,
}

n = y_test.shape[0]

results = {}

for model_name, (y_true, y_pred) in models.items():
    results[model_name] = {}
    for metric_name, metric_func in metrics.items():
        mean_val, pm_val = bootstrap_metric_ci_pm(y_true, y_pred, metric_func, n_bootstrap=n, random_state=42)
        results[model_name][metric_name] = (mean_val, pm_val)

In [18]:
print("\nBootstrap results (95% CI as ± half-width)")

for model_name, metric_results in results.items():
    print(f"\n{model_name}")
    for metric_name, (mean_val, pm_val) in metric_results.items():
        print(f"{metric_name}: {mean_val:.4f} ± {pm_val:.4f}")


Bootstrap results (95% CI as ± half-width)

Linear Regression
RMSE: 2.5034 ± 0.0343
R2: 0.9273 ± 0.0016
MAE: 1.7573 ± 0.0121

Random Forest
RMSE: 1.8836 ± 0.0322
R2: 0.9589 ± 0.0014
MAE: 1.2576 ± 0.0095

Neural Network
RMSE: 1.7144 ± 0.0321
R2: 0.9659 ± 0.0013
MAE: 1.1298 ± 0.0087
